In [ ]:
import pandas as pd 

df = pd.read_excel(r"C:\Users\User\Desktop\Thesis\IgAN374  .xls")

df.head()

In [ ]:
%pip install "xlrd>=2.0.1"

In [ ]:
df = pd.read_excel(r"C:\Users\User\Desktop\Thesis\IgAN374  .xls" )

print (df.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.info())

In [ ]:
print(df["The combined endpoint"].value_counts())

In [ ]:
print("Gender:")
print(df["Gender"].value_counts())

print("\nDisease course:")
print(df["Disease course"].value_counts())

print("\nHypertension History:")
print(df["Hypertension History"].value_counts())

print("\nHepatitis History:")
print(df["Hepatitis History"].value_counts())

print("\nRAAS blockade:")
print(df["RAAS blockade"].value_counts())

print("\nImmunosuppressant:")
print(df["Immunosuppressant"].value_counts())

In [ ]:
print(df.describe().T)

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
for col in [
    "OxfordM",
    "OxfordE",
    "OxfordS",
    "OxfordT",
    "OxfordC",
    "Global crescent",
    "Cellular global crescent",
    "Global crescent of fibrocyte",
    "Global fibrous crescent",
    "Segmental crescent",
    "Cellular segmental crescent",
    "Segmental crescent of fibrocyte",
    "Segmental fibrous crescent"
]:
    print(f"\n{col}")
    print(df[col].value_counts().sort_index())

In [ ]:
features = [
    "Baseline eGFR",
    "Serum creatine",
    "Triglyceride",
    "Proteinuria",
    "MAP",
    "Hematuria red blood cells/HPF",
    "Serum C3",
    "Age",
    "Crescent proportion",
    "Global crescent proportion"
]



print(df[features].corr(method="spearman"))

In [ ]:
print(df[features].skew().sort_values(ascending=False))

In [ ]:
print(df[features].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T)

In [ ]:
X = df[features]

y = df["The combined endpoint"]

print (X.shape)
print (y.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for feature in features:

    plt.figure(figsize=(7, 5))

    sns.boxplot(
        data=df,
        x="The combined endpoint",
        y=feature
    )

    plt.xlabel("Combined endpoint")
    plt.ylabel(feature)
    plt.title(f"{feature} by outcome")

    plt.show()

In [ ]:
for feature in features:

    plt.figure(figsize=(7, 5))

    sns.histplot(
        data=df,
        x=feature,
        hue="The combined endpoint",
        kde=True,
        element="step",
        stat="density",
        common_norm=False
    )

    plt.xlabel(feature)
    plt.ylabel("Density")
    plt.title(f"Distribution of {feature} by outcome")

    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.stats import shapiro

normality_results = []

for feature in features:

    class_0 = df.loc[df["The combined endpoint"] == 0, feature].dropna()
    class_1 = df.loc[df["The combined endpoint"] == 1, feature].dropna()

    stat_0, p_0 = shapiro(class_0)
    stat_1, p_1 = shapiro(class_1)

    normality_results.append({
        "Feature": feature,
        "Class 0 p-value": p_0,
        "Class 1 p-value": p_1
    })

normality_df = pd.DataFrame(normality_results)

print(normality_df.to_string(index=False))

In [ ]:
# δεν εχουμε κανονικότητα σε κανένα απο τα features άρα θα προχωρήσουμε με Mann-Whitney

from scipy.stats import mannwhitneyu

comparison_results = []

for feature in features:

    class_0 = df.loc[
        df["The combined endpoint"] == 0,
        feature
    ].dropna()

    class_1 = df.loc[
        df["The combined endpoint"] == 1,
        feature
    ].dropna()

    statistic, p_value = mannwhitneyu(
        class_0,
        class_1,
        alternative="two-sided"
    )

    comparison_results.append({
        "Feature": feature,
        "Class 0 Median": class_0.median(),
        "Class 1 Median": class_1.median(),
        "U statistic": statistic,
        "p-value": p_value
    })

comparison_df = pd.DataFrame(comparison_results)

print(comparison_df.to_string(index=False))

#Δεν παρατηρήθηκε στατιστικά σημαντική διαφορά στις univariate distributions των 10 predictors 
#μεταξύ των δύο outcome classes σύμφωνα με το Mann–Whitney U test.

In [ ]:
from statsmodels.stats.multitest import multipletests

comparison_df["FDR_p_value"] = multipletests(
    comparison_df["p-value"],
    method="fdr_bh"
)[1]

print(comparison_df.to_string(index=False))

#Με βάση αυτή την univariate σύγκριση, δεν έχουμε στατιστικά σημαντική διαφορά σε κανένα από τα 10 features
#μεταξύ των δύο outcome classes, ούτε πριν ούτε μετά τη διόρθωση για πολλαπλές συγκρίσεις.

In [ ]:
# ============================================
# DESCRIPTIVE STATISTICS
# Median (IQR) by outcome class
# ============================================

features = [
    "Baseline eGFR",
    "Serum creatine",
    "Triglyceride",
    "Proteinuria",
    "MAP",
    "Hematuria red blood cells/HPF",
    "Serum C3",
    "Age",
    "Crescent proportion",
    "Global crescent proportion"
]

descriptive_results = []

for feature in features:

    class_0 = df.loc[
        df["The combined endpoint"] == 0,
        feature
    ].dropna()

    class_1 = df.loc[
        df["The combined endpoint"] == 1,
        feature
    ].dropna()

    median_0 = class_0.median()
    q1_0 = class_0.quantile(0.25)
    q3_0 = class_0.quantile(0.75)
    iqr_0 = q3_0 - q1_0

    median_1 = class_1.median()
    q1_1 = class_1.quantile(0.25)
    q3_1 = class_1.quantile(0.75)
    iqr_1 = q3_1 - q1_1

    descriptive_results.append({
        "Feature": feature,
        "Class 0 Median": median_0,
        "Class 0 IQR": iqr_0,
        "Class 1 Median": median_1,
        "Class 1 IQR": iqr_1
    })

descriptive_df = pd.DataFrame(descriptive_results)

print(descriptive_df)

In [ ]:
final_comparison_df = pd.DataFrame({
    "Feature": comparison_df["Feature"],

    "Class 0 Median (IQR)": [
        f"{row['Class 0 Median']:.2f} ({row['Class 0 IQR']:.2f})"
        for _, row in descriptive_df.iterrows()
    ],

    "Class 1 Median (IQR)": [
        f"{row['Class 1 Median']:.2f} ({row['Class 1 IQR']:.2f})"
        for _, row in descriptive_df.iterrows()
    ],

    "U statistic": comparison_df["U statistic"].round(2),

    "p-value": comparison_df["p-value"].round(4),

    "FDR p-value": comparison_df["FDR_p_value"].round(4)
})

print(final_comparison_df.to_string(index=False))

#Το table 1 του άρθρου αναφέρει ακριβώς τα ίδια p-values 

In [ ]:
from sklearn.model_selection import train_test_split 

X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    stratify = y,
    random_state = 42
)

print ("Training set:" , X_train.shape)
print ("Test set:" , X_test.shape)
print (y_train.shape)
print (y_test.shape)

In [ ]:
print("Training outcome:")
print(y_train.value_counts())

print("\nTest outcome:")
print(y_test.value_counts())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(probability=True, random_state=42))
])

param_grid_svm = {
    "svm__C" : [0.01, 0.1, 1, 10, 100],
    "svm__gamma" : ["scale", 0.001, 0.01, 0.1, 1],
    "svm__kernel" : ["rbf", "linear"]
}

svm_grid = GridSearchCV (
    estimator=svm_pipeline,
    param_grid=param_grid_svm,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(svm_grid.best_params_)

In [ ]:
print("Best CV ROC-AUC:")
print(svm_grid.best_score_)

In [ ]:
final_svm_model = svm_grid.best_estimator_

y_pred_svm = final_svm_model.predict(X_test)

print (y_pred_svm)

In [ ]:
print (y_test.values)

In [ ]:
print("Predicted classes:")
print(pd.Series(y_pred_svm).value_counts())

In [ ]:
from sklearn.metrics import confusion_matrix

cm_svm = confusion_matrix(y_test, y_pred_svm)

print("Confusion Matrix:")
print(cm_svm)

In [ ]:
y_prob_svm = final_svm_model.predict_proba(X_test)[:, 1]

print (y_prob_svm)

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

print("Test ROC-AUC:", roc_auc_svm)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, zero_division=0)
recall_svm = recall_score(y_test, y_pred_svm, zero_division=0)
f1_svm = f1_score(y_test, y_pred_svm, zero_division=0)

print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)

In [ ]:
from sklearn.metrics import average_precision_score

pr_auc_svm = average_precision_score(y_test, y_prob_svm)

print("Test PR-AUC:", pr_auc_svm)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

param_grid_rf = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(grid_rf.best_params_)

In [ ]:
print("Best CV ROC-AUC:")
print(grid_rf.best_score_)

In [ ]:
final_rf_model = grid_rf.best_estimator_

y_pred_rf = final_rf_model.predict(X_test)

print (y_pred_rf)

In [ ]:
print("Predicted classes:")
print(pd.Series(y_pred_rf).value_counts())

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Confusion Matrix:")
print(cm_rf)

In [ ]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)

print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)

In [ ]:
y_prob_rf = final_rf_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob_rf)

print(y_prob_rf)

print("roc_auc_score_rf")
print(roc_auc)

In [ ]:
pr_auc_rf = average_precision_score(y_test, y_prob_rf)

print("Test PR-AUC:", pr_auc_rf)

In [ ]:
import numpy as np

import pandas as pd 

import matplotlib.pyplot as plt

from imblearn.over_sampling import RandomOverSampler

from imblearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix

svm_pipeline = Pipeline([
    ("oversampler" , RandomOverSampler(random_state=42)),
    ("scaler", StandardScaler()),
    ("svm", SVC(probability=True, random_state=42))
])

param_grid_svm = {
    "svm__C" : [0.01, 0.1, 1, 10, 100],
    "svm__gamma" : ["scale", 0.001, 0.01, 0.1, 1],
    "svm__kernel" : ["rbf", "linear"]
}

rf_pipeline = Pipeline([
    ("oversampler" , RandomOverSampler(random_state=42)),
    ("rf" , RandomForestClassifier(
        random_state = 42
    ))
])

param_grid_rf = {
    "rf__n_estimators": [100, 200, 500],
    "rf__max_depth": [None, 5, 10],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", "log2"]
}

logreg_pipeline = Pipeline([
    ("oversampler", RandomOverSampler(random_state=42)),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

param_grid_logreg = {
    "logreg__C": [0.01, 0.1, 1, 10, 100],
    "logreg__penalty": ["l1", "l2"],
    "logreg__solver": ["liblinear"]
}

In [ ]:
models = ["SVM", "RF", "Logistic Regression"]

Results_svm = []
Results_RF = []
Results_logreg = []

cms_svm = []
cms_rf = []
cms_logreg = []

threshold_results_svm = []
threshold_results_rf = []
threshold_results_logreg = []

for i in range(20):

    X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    stratify = y,
    random_state = i
        
    )
    
    for model in models:

        if model == "SVM":

            svm_grid = GridSearchCV (
                estimator=svm_pipeline,
                param_grid=param_grid_svm,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1
        )

            svm_grid.fit(X_train, y_train)

            best_svm = svm_grid.best_estimator_

            y_pred_svm = best_svm.predict(X_test)

            cms_svm.append(confusion_matrix(y_test, y_pred_svm))

            y_prob_svm = best_svm.predict_proba(X_test)[:, 1]

            thresholds = np.arange(0.1, 1.0, 0.1)

            for threshold in thresholds:
                
                y_pred_threshold = (y_prob_svm >= threshold).astype(int)
                
                accuracy_t = accuracy_score(y_test, y_pred_threshold)
                precision_t = precision_score(y_test, y_pred_threshold, zero_division=0)
                recall_t = recall_score(y_test, y_pred_threshold, zero_division=0)
                f1_t = f1_score(y_test, y_pred_threshold, zero_division=0)
                cm_t = confusion_matrix(y_test, y_pred_threshold)
                TN_t, FP_t, FN_t, TP_t = cm_t.ravel()
                
                specificity_t = TN_t / (TN_t + FP_t)

                threshold_results_svm.append({
                    "Iteration": i + 1,
                    "Threshold": threshold,
                    "Accuracy": accuracy_t,
                    "Precision": precision_t,
                    "Recall": recall_t,
                    "Specificity": specificity_t,
                    "F1": f1_t
                })

    
        
    
            accuracy = accuracy_score(y_test, y_pred_svm)
            precision = precision_score(y_test, y_pred_svm, zero_division=0)
            recall = recall_score(y_test, y_pred_svm, zero_division=0)
            f1 = f1_score(y_test, y_pred_svm, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_prob_svm)
            pr_auc = average_precision_score(y_test, y_prob_svm)

            cm = confusion_matrix(y_test, y_pred_svm)
            TN, FP, FN, TP = cm.ravel()

            specificity = TN / (TN + FP)

            Results_svm.append({
                "iteration": i + 1,
                "Accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc-auc": roc_auc,
                "pr-auc": pr_auc,
                "Specificity": specificity
            })
        
        elif model == "RF":
            
            rf_grid = GridSearchCV(
                estimator=rf_pipeline,
                param_grid=param_grid_rf,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1
            )

            rf_grid.fit(X_train, y_train)

            best_rf = rf_grid.best_estimator_

            y_pred_rf = best_rf.predict(X_test)

            cms_rf.append(confusion_matrix(y_test, y_pred_rf))
            
            y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

            thresholds = np.arange(0.1, 1.0, 0.1)

            for threshold in thresholds:
                
                y_pred_threshold = (y_prob_rf >= threshold).astype(int)

                accuracy_t = accuracy_score(y_test, y_pred_threshold)
                precision_t = precision_score(y_test, y_pred_threshold, zero_division=0)
                recall_t = recall_score(y_test, y_pred_threshold, zero_division=0)
                f1_t = f1_score(y_test, y_pred_threshold, zero_division=0)

                cm_t = confusion_matrix(y_test, y_pred_threshold)
                TN_t, FP_t, FN_t, TP_t = cm_t.ravel()

                specificity_t = TN_t / (TN_t + FP_t)

                threshold_results_rf.append({
                    "Iteration": i + 1,
                    "Threshold": threshold,
                    "Accuracy": accuracy_t,
                    "Precision": precision_t,
                    "Recall": recall_t,
                    "Specificity": specificity_t,
                    "F1": f1_t
                })

            accuracy = accuracy_score(y_test, y_pred_rf)
            precision = precision_score(y_test, y_pred_rf, zero_division=0)
            recall = recall_score(y_test, y_pred_rf, zero_division=0)
            f1 = f1_score(y_test, y_pred_rf, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_prob_rf)
            pr_auc = average_precision_score(y_test, y_prob_rf)

            cm = confusion_matrix(y_test, y_pred_rf)
            TN, FP, FN, TP = cm.ravel()

            specificity = TN / (TN + FP)

            Results_RF.append({
                "iteration": i + 1,
                "Accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc-auc": roc_auc,
                "pr-auc": pr_auc,
                "Specificity": specificity
            })


        elif model == "Logistic Regression":

            logreg_grid = GridSearchCV (
                estimator=logreg_pipeline,
                param_grid=param_grid_logreg,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1
        )
            logreg_grid.fit(X_train, y_train)

            best_logreg = logreg_grid.best_estimator_

            y_pred_logreg = best_logreg.predict(X_test)

            cms_logreg.append(confusion_matrix(y_test, y_pred_logreg))

            y_prob_logreg = best_logreg.predict_proba(X_test)[:, 1]
            
            thresholds = np.arange(0.1, 1.0, 0.1)

            for threshold in thresholds:
                
                y_pred_threshold = (y_prob_logreg >= threshold).astype(int)

                accuracy_t = accuracy_score(y_test, y_pred_threshold)
                precision_t = precision_score(y_test, y_pred_threshold, zero_division=0)
                recall_t = recall_score(y_test, y_pred_threshold, zero_division=0)
                f1_t = f1_score(y_test, y_pred_threshold, zero_division=0)

                cm_t = confusion_matrix(y_test, y_pred_threshold)
                TN_t, FP_t, FN_t, TP_t = cm_t.ravel()

                specificity_t = TN_t / (TN_t + FP_t)

                threshold_results_logreg.append({
                    "Iteration": i + 1,
                    "Threshold": threshold,
                    "Accuracy": accuracy_t,
                    "Precision": precision_t,
                    "Recall": recall_t,
                    "Specificity": specificity_t,
                    "F1": f1_t
                })

            accuracy = accuracy_score(y_test, y_pred_logreg)
            precision = precision_score(y_test, y_pred_logreg, zero_division=0)
            recall = recall_score(y_test, y_pred_logreg, zero_division=0)
            f1 = f1_score(y_test, y_pred_logreg, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_prob_logreg)
            pr_auc = average_precision_score(y_test, y_prob_logreg)

            cm = confusion_matrix(y_test, y_pred_logreg)
            TN, FP, FN, TP = cm.ravel()

            specificity = TN / (TN + FP)

            Results_logreg.append({
                "iteration": i + 1,
                "Accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc-auc": roc_auc,
                "pr-auc": pr_auc,
                "Specificity": specificity
            })

            

            

            
df_svm = pd.DataFrame(Results_svm)
df_RF = pd.DataFrame(Results_RF)
df_logreg = pd.DataFrame(Results_logreg)

df_threshold_svm = pd.DataFrame(threshold_results_svm)
df_threshold_rf = pd.DataFrame(threshold_results_rf)
df_threshold_logreg = pd.DataFrame(threshold_results_logreg)

total_cm_svm = np.sum(cms_svm, axis=0)
total_cm_rf = np.sum(cms_rf, axis=0)
total_cm_logreg = np.sum(cms_logreg, axis=0)

print("SVM:")
print (df_svm)

print("\nRandom Forest:")
print (df_RF)

print("\nLogistic Regression:")
print (df_logreg)

print("SVM:")
print(total_cm_svm)

print("\nRandom Forest:")
print(total_cm_rf)

print("\nLogistic Regression:")
print(total_cm_logreg)

print(df_threshold_svm.shape)
print(df_threshold_rf.shape)
print(df_threshold_logreg.shape)
            

In [ ]:
mean_results_svm = df_svm[["Accuracy", "precision", "recall", "f1", "roc-auc", "pr-auc", "Specificity"]].mean()

std_results_svm = df_svm[["Accuracy", "precision", "recall", "f1", "roc-auc", "pr-auc", "Specificity"]].std()

mean_results_RF = df_RF[["Accuracy", "precision", "recall", "f1", "roc-auc", "pr-auc", "Specificity"]].mean()

std_results_RF = df_RF[["Accuracy", "precision", "recall", "f1", "roc-auc", "pr-auc", "Specificity"]].std()

mean_results_logreg = df_logreg[["Accuracy", "precision", "recall", "f1", "roc-auc", "pr-auc", "Specificity"]].mean()

std_results_logreg = df_logreg[["Accuracy", "precision", "recall", "f1", "roc-auc", "pr-auc", "Specificity"]].std()

print("mean_svm:")
print(mean_results_svm)

print("\nstandard_deviation_svm:")
print(std_results_svm)

print("mean_RF:")
print(mean_results_RF)

print("\nstandard_deviation_RF:")
print(std_results_RF)

print("mean_logreg:")
print(mean_results_logreg)

print("\nstandard_deviation_logreg:")
print(std_results_logreg)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    df_svm["iteration"],
    df_svm["roc-auc"],
    marker="o",
    label="SVM"
)

plt.plot(
    df_RF["iteration"],
    df_RF["roc-auc"],
    marker="o",
    label="Random Forest"
)

plt.plot(
    df_logreg["iteration"],
    df_logreg["roc-auc"],
    marker="o",
    label="Logistic Regression"
)

plt.xlabel("Iteration")
plt.ylabel("ROC-AUC")
plt.title("ROC-AUC across 20 iterations")
plt.xticks(range(1, 21))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    df_svm["iteration"],
    df_svm["pr-auc"],
    marker="o",
    label="SVM"
)

plt.plot(
    df_RF["iteration"],
    df_RF["pr-auc"],
    marker="o",
    label="Random Forest"
)

plt.plot(
    df_logreg["iteration"],
    df_logreg["pr-auc"],
    marker="o",
    label="Logistic Regression"
)

plt.xlabel("Iteration")
plt.ylabel("PR-AUC")
plt.title("PR-AUC across 20 iterations")
plt.xticks(range(1, 21))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
comparison_table = pd.DataFrame({
    "Model": ["SVM", "Random Forest", "Logistic Regression"],
    
    "Accuracy": [
        f"{mean_results_svm['Accuracy']:.3f} ± {std_results_svm['Accuracy']:.3f}",
        f"{mean_results_RF['Accuracy']:.3f} ± {std_results_RF['Accuracy']:.3f}",
        f"{mean_results_logreg['Accuracy']:.3f} ± {std_results_logreg['Accuracy']:.3f}"
    ],
    
    "Precision": [
        f"{mean_results_svm['precision']:.3f} ± {std_results_svm['precision']:.3f}",
        f"{mean_results_RF['precision']:.3f} ± {std_results_RF['precision']:.3f}",
        f"{mean_results_logreg['precision']:.3f} ± {std_results_logreg['precision']:.3f}"
    ],
    
    "Recall": [
        f"{mean_results_svm['recall']:.3f} ± {std_results_svm['recall']:.3f}",
        f"{mean_results_RF['recall']:.3f} ± {std_results_RF['recall']:.3f}",
        f"{mean_results_logreg['recall']:.3f} ± {std_results_logreg['recall']:.3f}"
    ],
    
    "Specificity": [
    f"{mean_results_svm['Specificity']:.3f} ± {std_results_svm['Specificity']:.3f}",
    f"{mean_results_RF['Specificity']:.3f} ± {std_results_RF['Specificity']:.3f}",
    f"{mean_results_logreg['Specificity']:.3f} ± {std_results_logreg['Specificity']:.3f}"
    ],
    
    "F1": [
        f"{mean_results_svm['f1']:.3f} ± {std_results_svm['f1']:.3f}",
        f"{mean_results_RF['f1']:.3f} ± {std_results_RF['f1']:.3f}",
        f"{mean_results_logreg['f1']:.3f} ± {std_results_logreg['f1']:.3f}"
    ],
    
    "ROC-AUC": [
        f"{mean_results_svm['roc-auc']:.3f} ± {std_results_svm['roc-auc']:.3f}",
        f"{mean_results_RF['roc-auc']:.3f} ± {std_results_RF['roc-auc']:.3f}",
        f"{mean_results_logreg['roc-auc']:.3f} ± {std_results_logreg['roc-auc']:.3f}"
    ],
    
    "PR-AUC": [
        f"{mean_results_svm['pr-auc']:.3f} ± {std_results_svm['pr-auc']:.3f}",
        f"{mean_results_RF['pr-auc']:.3f} ± {std_results_RF['pr-auc']:.3f}",
        f"{mean_results_logreg['pr-auc']:.3f} ± {std_results_logreg['pr-auc']:.3f}"
    ]
})

comparison_table

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "Specificity", "F1", "ROC-AUC", "PR-AUC"]

svm_means = [
    mean_results_svm["Accuracy"],
    mean_results_svm["precision"],
    mean_results_svm["recall"],
    mean_results_svm["Specificity"],
    mean_results_svm["f1"],
    mean_results_svm["roc-auc"],
    mean_results_svm["pr-auc"]
]

svm_std = [
    std_results_svm["Accuracy"],
    std_results_svm["precision"],
    std_results_svm["recall"],
    std_results_svm["Specificity"],
    std_results_svm["f1"],
    std_results_svm["roc-auc"],
    std_results_svm["pr-auc"]
]

rf_means = [
    mean_results_RF["Accuracy"],
    mean_results_RF["precision"],
    mean_results_RF["recall"],
    mean_results_RF["Specificity"],
    mean_results_RF["f1"],
    mean_results_RF["roc-auc"],
    mean_results_RF["pr-auc"]
]

rf_std = [
    std_results_RF["Accuracy"],
    std_results_RF["precision"],
    std_results_RF["recall"],
    std_results_RF["Specificity"],
    std_results_RF["f1"],
    std_results_RF["roc-auc"],
    std_results_RF["pr-auc"]
]

logreg_means = [
    mean_results_logreg["Accuracy"],
    mean_results_logreg["precision"],
    mean_results_logreg["recall"],
    mean_results_logreg["Specificity"],
    mean_results_logreg["f1"],
    mean_results_logreg["roc-auc"],
    mean_results_logreg["pr-auc"]
]

logreg_std = [
    std_results_logreg["Accuracy"],
    std_results_logreg["precision"],
    std_results_logreg["recall"],
    std_results_logreg["Specificity"],
    std_results_logreg["f1"],
    std_results_logreg["roc-auc"],
    std_results_logreg["pr-auc"]
]

x = np.arange(len(metrics))
width = 0.25

plt.figure(figsize=(12, 6))

plt.bar(
    x - width,
    svm_means,
    width,
    yerr=svm_std,
    capsize=4,
    label="SVM"
)

plt.bar(
    x,
    rf_means,
    width,
    yerr=rf_std,
    capsize=4,
    label="Random Forest"
)

plt.bar(
    x + width,
    logreg_means,
    width,
    yerr=logreg_std,
    capsize=4,
    label="Logistic Regression"
)

plt.xlabel("Performance Metric")
plt.ylabel("Mean Score")
plt.title("Model Performance: Mean ± SD across 20 iterations")

plt.xticks(x, metrics)
plt.ylim(0, 1)

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
svm_threshold_summary = (
    df_threshold_svm
    .groupby("Threshold")
    .agg({
        "Accuracy": ["mean", "std"],
        "Precision": ["mean", "std"],
        "Recall": ["mean", "std"],
        "Specificity": ["mean", "std"],
        "F1": ["mean", "std"]
    })
)

rf_threshold_summary = (
    df_threshold_rf
    .groupby("Threshold")
    .agg({
        "Accuracy": ["mean", "std"],
        "Precision": ["mean", "std"],
        "Recall": ["mean", "std"],
        "Specificity": ["mean", "std"],
        "F1": ["mean", "std"]
    })
)

logreg_threshold_summary = (
    df_threshold_logreg
    .groupby("Threshold")
    .agg({
        "Accuracy": ["mean", "std"],
        "Precision": ["mean", "std"],
        "Recall": ["mean", "std"],
        "Specificity": ["mean", "std"],
        "F1": ["mean", "std"]
    })
)

print("SVM:")
print(svm_threshold_summary)

print("\nRandom Forest:")
print(rf_threshold_summary)

print("\nLogistic Regression:")
print(logreg_threshold_summary)

In [ ]:
import matplotlib.pyplot as plt

thresholds_plot = svm_threshold_summary.index

recall_mean = svm_threshold_summary[("Recall", "mean")]
specificity_mean = svm_threshold_summary[("Precision", "mean")]

plt.figure(figsize=(8, 5))

plt.plot(
    thresholds_plot,
    recall_mean,
    marker="o",
    label="Recall"
)

plt.plot(
    thresholds_plot,
    specificity_mean,
    marker="o",
    label="Precision"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("SVM: Recall and Precision across Classification Thresholds")

plt.xticks(thresholds_plot)
plt.ylim(0, 1.05)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
thresholds_plot = rf_threshold_summary.index

recall_mean = rf_threshold_summary[("Recall", "mean")]
specificity_mean = rf_threshold_summary[("Precision", "mean")]

plt.figure(figsize=(8, 5))

plt.plot(
    thresholds_plot,
    recall_mean,
    marker="o",
    label="Recall"
)

plt.plot(
    thresholds_plot,
    specificity_mean,
    marker="o",
    label="Precision"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Random Forest: Recall and Precision across Classification Thresholds")

plt.xticks(thresholds_plot)
plt.ylim(0, 1.05)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
thresholds_plot = logreg_threshold_summary.index

recall_mean = logreg_threshold_summary[("Recall", "mean")]
specificity_mean = logreg_threshold_summary[("Precision", "mean")]

plt.figure(figsize=(8, 5))

plt.plot(
    thresholds_plot,
    recall_mean,
    marker="o",
    label="Recall"
)

plt.plot(
    thresholds_plot,
    specificity_mean,
    marker="o",
    label="Precision"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Logistic Regression: Recall and Precision across Classification Thresholds")

plt.xticks(thresholds_plot)
plt.ylim(0, 1.05)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    svm_threshold_summary.index,
    svm_threshold_summary[("F1", "mean")],
    marker="o",
    label="SVM"
)

plt.plot(
    rf_threshold_summary.index,
    rf_threshold_summary[("F1", "mean")],
    marker="o",
    label="Random Forest"
)

plt.plot(
    logreg_threshold_summary.index,
    logreg_threshold_summary[("F1", "mean")],
    marker="o",
    label="Logistic Regression"
)

plt.xlabel("Classification Threshold")
plt.ylabel("F1-score")
plt.title("F1-score across Classification Thresholds")

plt.xticks(np.arange(0.1, 1.0, 0.1))
plt.ylim(0, 0.4)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

thresholds_plot = svm_threshold_summary.index

precision_mean = svm_threshold_summary[("Precision", "mean")]
recall_mean = svm_threshold_summary[("Recall", "mean")]
f1_mean = svm_threshold_summary[("F1", "mean")]

plt.figure(figsize=(8, 5))

plt.plot(
    thresholds_plot,
    precision_mean,
    marker="o",
    label="Precision"
)

plt.plot(
    thresholds_plot,
    recall_mean,
    marker="o",
    label="Recall"
)

plt.plot(
    thresholds_plot,
    f1_mean,
    marker="o",
    label="F1-score"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("SVM: Precision, Recall and F1-score across Classification Thresholds")

plt.xticks(thresholds_plot)
plt.ylim(0, 1.05)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

thresholds_plot = rf_threshold_summary.index

precision_mean = rf_threshold_summary[("Precision", "mean")]
recall_mean = rf_threshold_summary[("Recall", "mean")]
f1_mean = rf_threshold_summary[("F1", "mean")]

plt.figure(figsize=(8, 5))

plt.plot(
    thresholds_plot,
    precision_mean,
    marker="o",
    label="Precision"
)

plt.plot(
    thresholds_plot,
    recall_mean,
    marker="o",
    label="Recall"
)

plt.plot(
    thresholds_plot,
    f1_mean,
    marker="o",
    label="F1-score"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Random Forest: Precision, Recall and F1-score across Classification Thresholds")

plt.xticks(thresholds_plot)
plt.ylim(0, 1.05)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

thresholds_plot = logreg_threshold_summary.index

precision_mean = logreg_threshold_summary[("Precision", "mean")]
recall_mean = logreg_threshold_summary[("Recall", "mean")]
f1_mean = logreg_threshold_summary[("F1", "mean")]

plt.figure(figsize=(8, 5))

plt.plot(
    thresholds_plot,
    precision_mean,
    marker="o",
    label="Precision"
)

plt.plot(
    thresholds_plot,
    recall_mean,
    marker="o",
    label="Recall"
)

plt.plot(
    thresholds_plot,
    f1_mean,
    marker="o",
    label="F1-score"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Logistic Regression: Precision, Recall and F1-score across Classification Thresholds")

plt.xticks(thresholds_plot)
plt.ylim(0, 1.05)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import normalized_mutual_info_score


# ---------------------------------------------------------
# The 10 predictors used in the models
# ---------------------------------------------------------

features = [
    "Baseline eGFR",
    "Serum creatine",
    "Triglyceride",
    "Proteinuria",
    "MAP",
    "Hematuria red blood cells/HPF",
    "Serum C3",
    "Age",
    "Crescent proportion",
    "Global crescent proportion"
]


# ---------------------------------------------------------
# Copy of the predictors
# ---------------------------------------------------------

X_nmi = df[features].copy()


# ---------------------------------------------------------
# Continuous variables into 5 quantile bins
# ---------------------------------------------------------

from sklearn.preprocessing import KBinsDiscretizer

# Discretization of continuous predictors
# into up to 5 quantile-based bins

discretizer = KBinsDiscretizer(
    n_bins=5,
    encode="ordinal",
    strategy="quantile",
    subsample=None
)

X_binned_array = discretizer.fit_transform(X_nmi)

X_binned = pd.DataFrame(
    X_binned_array,
    columns=features,
    index=X_nmi.index
)


global_crescent = X_nmi["Global crescent proportion"]


global_binned = pd.Series(
    0,
    index=global_crescent.index,
    dtype=float
)


positive_mask = global_crescent > 0


positive_values = global_crescent[positive_mask]

positive_bins = pd.qcut(
    positive_values,
    q=4,
    labels=False,
    duplicates="drop"
)


global_binned.loc[positive_mask] = positive_bins.values + 1


X_binned["Global crescent proportion"] = global_binned


# ---------------------------------------------------------
# Calculate NMI between every pair of predictors
# ---------------------------------------------------------

n_features = len(features)

nmi_matrix = np.zeros((n_features, n_features))

for i in range(n_features):
    for j in range(n_features):

        valid = (
            X_binned.iloc[:, i].notna()
            & X_binned.iloc[:, j].notna()
        )

        nmi_matrix[i, j] = normalized_mutual_info_score(
        X_binned.loc[valid, features[i]],
        X_binned.loc[valid, features[j]]
        )


nmi_df = pd.DataFrame(
    nmi_matrix,
    index=features,
    columns=features
)


# ---------------------------------------------------------
# Plot heatmap
# ---------------------------------------------------------

plt.figure(figsize=(11, 9))

im = plt.imshow(
    nmi_df.values,
    cmap="viridis",
    vmin=0,
    vmax=1
)

plt.colorbar(im, label="Normalized Mutual Information")

plt.xticks(
    range(n_features),
    features,
    rotation=90
)

plt.yticks(
    range(n_features),
    features
)


for i in range(n_features):
    for j in range(n_features):
        plt.text(
            j,
            i,
            f"{nmi_df.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title(
    "Normalized Mutual Information between Predictors"
)

plt.tight_layout()

plt.savefig(
    "NMI_Features_Heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ---------------------------------------------------------
# NMI between each predictor and the combined endpoint
# ---------------------------------------------------------

target = "The combined endpoint"

y = df[target]

nmi_target = {}

for feature in features:

    valid = (
        X_binned[feature].notna()
        & y.notna()
    )

    nmi_target[feature] = normalized_mutual_info_score(
        X_binned.loc[valid, feature],
        y.loc[valid]
    )

nmi_target_df = pd.DataFrame(
    list(nmi_target.items()),
    columns=["Feature", "NMI"]
)

# Sort from highest to lowest
nmi_target_df = nmi_target_df.sort_values(
    "NMI",
    ascending=False
)

print(nmi_target_df)

In [ ]:
# ---------------------------------------------------------
# NMI heatmap: Predictors vs Combined Endpoint
# ---------------------------------------------------------

plt.figure(figsize=(7, 8))

matrix = nmi_target_df.set_index("Feature")[["NMI"]]

im = plt.imshow(
    matrix.values,
    cmap="viridis",
    vmin=0,
    vmax=matrix["NMI"].max(),
    aspect="auto"
)

plt.colorbar(
    im,
    label="Normalized Mutual Information"
)

plt.xticks(
    [0],
    ["Combined Endpoint"]
)

plt.yticks(
    range(len(matrix)),
    matrix.index
)


for i, value in enumerate(matrix["NMI"]):
    plt.text(
        0,
        i,
        f"{value:.3f}",
        ha="center",
        va="center",
        color="white" if value > 0.5 else "black",
        fontsize=9
    )

plt.title(
    "Normalized Mutual Information between Predictors and Combined Endpoint"
)

plt.tight_layout()

plt.savefig(
    "NMI_Features_vs_Endpoint.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ---------------------------------------------------------
# NMI heatmap: Predictors vs the two outcome classes
# ---------------------------------------------------------

target = "The combined endpoint"


none_endpoint = (df[target] == 0).astype(int)
endpoint = (df[target] == 1).astype(int)

nmi_none = {}
nmi_endpoint = {}

for feature in features:

    valid = X_binned[feature].notna()

    
    nmi_none[feature] = normalized_mutual_info_score(
        X_binned.loc[valid, feature],
        none_endpoint.loc[valid]
    )

  
    nmi_endpoint[feature] = normalized_mutual_info_score(
        X_binned.loc[valid, feature],
        endpoint.loc[valid]
    )


nmi_classes_df = pd.DataFrame({
    "None-Endpoint (0)": nmi_none,
    "Endpoint (1)": nmi_endpoint
})


nmi_classes_df = nmi_classes_df.loc[features]


plt.figure(figsize=(8, 8))

sns.heatmap(
    nmi_classes_df,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    vmin=0,
    vmax=nmi_classes_df.values.max(),
    cbar_kws={"label": "Normalized Mutual Information"}
)

plt.title(
    "Normalized Mutual Information between Predictors and Outcome Classes"
)

plt.xlabel("Combined Endpoint Class")
plt.ylabel("Predictor")

plt.tight_layout()

plt.savefig(
    "NMI_Predictors_vs_Two_Outcome_Classes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------------
# 1. ΡΥΘΜΙΣΕΙΣ ΔΕΔΟΜΕΝΩΝ
# --------------------------------------------------

target = "The combined endpoint"

predictors = [
   "Baseline eGFR",
    "Serum creatine",
    "Triglyceride",
    "Proteinuria",
    "MAP",
    "Hematuria red blood cells/HPF",
    "Serum C3",
    "Age",
    "Crescent proportion",
    "Global crescent proportion"
]

# Αν τα ονόματα των στηλών στο DataFrame σου διαφέρουν,
# αντικατάστησέ τα με τα ακριβή ονόματα από το df.columns.

# --------------------------------------------------
# 2. ENTROPY
# --------------------------------------------------

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log(p))


# --------------------------------------------------
# 3. DISCRETIZATION
# --------------------------------------------------

def discretize_predictor(x, n_bins=5, continuous_threshold=10,
                         is_global_crescent=False):

    x = x.copy()

    # Ειδικός χειρισμός Global crescent proportion
    if is_global_crescent:
        result = pd.Series(index=x.index, dtype=object)
        missing = x.isna()
        zero = x.eq(0) & ~missing
        positive = x.gt(0) & ~missing

        result.loc[zero] = "Zero"
        result.loc[missing] = "Missing"

        if positive.sum() > 0:
            if x.loc[positive].nunique() > 1:
                bins = min(n_bins, x.loc[positive].nunique())
                result.loc[positive] = (
                    pd.qcut(
                        x.loc[positive],
                        q=bins,
                        labels=False,
                        duplicates="drop"
                    ).astype(str).radd("Positive_")
                )
            else:
                result.loc[positive] = "Positive_1"

        return result

    # Διακριτοποίηση των συνεχών μεταβλητών
    if (pd.api.types.is_numeric_dtype(x)
            and x.nunique(dropna=True) > continuous_threshold):

        nonmissing = x.notna()
        result = pd.Series(index=x.index, dtype=object)

        if x.loc[nonmissing].nunique() > 1:
            bins = min(n_bins, x.loc[nonmissing].nunique())
            result.loc[nonmissing] = pd.qcut(
                x.loc[nonmissing],
                q=bins,
                labels=False,
                duplicates="drop"
            ).astype(str)

        else:
            result.loc[nonmissing] = "0"

        result.loc[~nonmissing] = "Missing"
        return result

    return x.astype(object).where(x.notna(), "Missing")


# --------------------------------------------------
# 4. CLASS-SPECIFIC MI CONTRIBUTIONS
# --------------------------------------------------

def nmi_per_class(x, y, n_bins=5,
                  continuous_threshold=10,
                  is_global_crescent=False):

    df_temp = pd.DataFrame({
        "x": x,
        "y": y
    }).dropna(subset=["y"]).copy()

    # Κρατάμε μόνο τις δύο αναμενόμενες κλάσεις
    df_temp = df_temp[df_temp["y"].isin([0, 1])]

    if df_temp.empty or df_temp["y"].nunique() < 2:
        return 0.0, 0.0, 0.0

    df_temp["x"] = discretize_predictor(
        df_temp["x"],
        n_bins=n_bins,
        continuous_threshold=continuous_threshold,
        is_global_crescent=is_global_crescent
    )

    # Κοινή κατανομή P(X, Y)
    p_xy = pd.crosstab(
        df_temp["x"],
        df_temp["y"],
        normalize=True
    ).reindex(columns=[0, 1], fill_value=0)

    p_x = p_xy.sum(axis=1).to_numpy()
    p_y = p_xy.sum(axis=0).to_numpy()

    # Συνεισφορά κάθε κλάσης στο MI
    contributions = {}

    for c in [0, 1]:
        p_xc = p_xy[c].to_numpy()
        mask = p_xc > 0

        contributions[c] = np.sum(
            p_xc[mask] *
            np.log(
                p_xc[mask] /
                (p_x[mask] * p_y[c])
            )
        )

    # Κοινός παρονομαστής: γεωμετρικός μέσος entropies
    denom = np.sqrt(entropy(p_x) * entropy(p_y))

    if denom == 0:
        return 0.0, 0.0, 0.0

    nmi_0 = contributions[0] / denom
    nmi_1 = contributions[1] / denom
    nmi_total = nmi_0 + nmi_1

    return nmi_0, nmi_1, nmi_total


# --------------------------------------------------
# 5. ΥΠΟΛΟΓΙΣΜΟΣ ΓΙΑ ΤΟΥΣ 10 PREDICTORS
# --------------------------------------------------

def nmi_table(df, target, predictors):

    rows = []

    for col in predictors:

        if col not in df.columns:
            raise ValueError(
                f"Δεν βρέθηκε η στήλη '{col}'. "
                "Έλεγξε τα ονόματα με print(df.columns.tolist())."
            )

        n0, n1, total = nmi_per_class(
            df[col],
            df[target],
            n_bins=5,
            continuous_threshold=10,
            is_global_crescent=(
                col == "Global crescent proportion of glomeruli"
            )
        )

        rows.append({
            "Predictor": col,
            "Class 0: None-Endpoint": n0,
            "Class 1: Endpoint": n1,
            "Total NMI": total
        })

    return (
        pd.DataFrame(rows)
        .sort_values("Total NMI", ascending=False)
        .reset_index(drop=True)
    )


# --------------------------------------------------
# 6. ΕΚΤΕΛΕΣΗ
# --------------------------------------------------

result = nmi_table(df, target, predictors)

display(result.round(5))

# Αποθήκευση αποτελεσμάτων
result.to_csv("IgAN_NMI_class_contributions.csv", index=False)


# --------------------------------------------------
# 7. HEATMAP ΜΕ ΔΥΟ ΣΤΗΛΕΣ
# --------------------------------------------------

heatmap_data = (
    result.set_index("Predictor")[
        ["Class 0: None-Endpoint", "Class 1: Endpoint"]
    ]
)

plt.figure(figsize=(8, 7))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".4f",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "Normalized MI contribution"}
)

plt.title("Class-specific contributions to Mutual Information")
plt.xlabel("Combined endpoint class")
plt.ylabel("Predictor")
plt.tight_layout()
plt.savefig(
    "IgAN_NMI_class_contributions_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
print(df.columns.tolist())


In [ ]:
for col in df.columns:
    if "endpoint" in str(col).lower():
        print(f"\nΣτήλη: {col}")
        print(df[col].value_counts(dropna=False))

In [ ]:
best_threshold_rows = []

for model_name, summary in [
    ("SVM", svm_threshold_summary),
    ("Random Forest", rf_threshold_summary),
    ("Logistic Regression", logreg_threshold_summary)
]:

    best_threshold = summary[("F1", "mean")].idxmax()

    best_threshold_rows.append({
        "Model": model_name,
        "Threshold": best_threshold,
        "F1": summary.loc[best_threshold, ("F1", "mean")],
        "Precision": summary.loc[best_threshold, ("Precision", "mean")],
        "Recall": summary.loc[best_threshold, ("Recall", "mean")],
        "Specificity": summary.loc[best_threshold, ("Specificity", "mean")]
    })

best_threshold_table = pd.DataFrame(best_threshold_rows)

print(best_threshold_table)

In [ ]:
for summary in [
    svm_threshold_summary,
    rf_threshold_summary,
    logreg_threshold_summary
]:
    
    summary[("Youden_J", "mean")] = (
        summary[("Recall", "mean")] +
        summary[("Specificity", "mean")] -
        1
    )



In [ ]:
youden_rows = []

for model_name, summary in [
    ("SVM", svm_threshold_summary),
    ("Random Forest", rf_threshold_summary),
    ("Logistic Regression", logreg_threshold_summary)
]:

    best_threshold = summary[("Youden_J", "mean")].idxmax()

    youden_rows.append({
        "Model": model_name,
        "Threshold": best_threshold,
        "Youden_J": summary.loc[best_threshold, ("Youden_J", "mean")],
        "Recall": summary.loc[best_threshold, ("Recall", "mean")],
        "Specificity": summary.loc[best_threshold, ("Specificity", "mean")],
        "Precision": summary.loc[best_threshold, ("Precision", "mean")],
        "F1": summary.loc[best_threshold, ("F1", "mean")]
    })

youden_table = pd.DataFrame(youden_rows)

print(youden_table)

In [ ]:
for summary in [
    svm_threshold_summary,
    rf_threshold_summary,
    logreg_threshold_summary
]:
    
    summary[("Balanced_Accuracy", "mean")] = (
        summary[("Recall", "mean")] +
        summary[("Specificity", "mean")]
    ) / 2

In [ ]:
balanced_rows = []

for model_name, summary in [
    ("SVM", svm_threshold_summary),
    ("Random Forest", rf_threshold_summary),
    ("Logistic Regression", logreg_threshold_summary)
]:

    best_threshold = summary[("Balanced_Accuracy", "mean")].idxmax()

    balanced_rows.append({
        "Model": model_name,
        "Threshold": best_threshold,
        "Balanced_Accuracy": summary.loc[
            best_threshold, ("Balanced_Accuracy", "mean")
        ],
        "Recall": summary.loc[best_threshold, ("Recall", "mean")],
        "Specificity": summary.loc[best_threshold, ("Specificity", "mean")],
        "Precision": summary.loc[best_threshold, ("Precision", "mean")],
        "F1": summary.loc[best_threshold, ("F1", "mean")]
    })

balanced_table = pd.DataFrame(balanced_rows)

print(balanced_table)

In [ ]:
import shap

print(shap.__version__)

In [ ]:
models_for_shap = {
    "SVM": best_svm,
    "Random Forest": best_rf,
    "Logistic Regression": best_logreg
}

for name, model in models_for_shap.items():
    print(name)
    print(type(model))
    print(model)
    print()

In [ ]:
# ==========================================
# SHAP ANALYSIS - 20 ITERATIONS
# ==========================================

SHAP_results = {
    "SVM": [],
    "RF": [],
    "Logistic Regression": []
}

SHAP_X_test = {
    "SVM": [],
    "RF": [],
    "Logistic Regression": []
}

for i in range(20):

    print(f"\n========== Iteration {i + 1}/20 ==========")

    # ------------------------------------------
    # Same train/test split as the main analysis
    # ------------------------------------------

    X_train_shap, X_test_shap, y_train_shap, y_test_shap = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=i
    )

    # ==========================================
    # SVM
    # ==========================================

    svm_grid_shap = GridSearchCV(
        estimator=svm_pipeline,
        param_grid=param_grid_svm,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1
    )

    svm_grid_shap.fit(X_train_shap, y_train_shap)

    best_svm_shap = svm_grid_shap.best_estimator_

    print("SVM fitted.")

    # SHAP explainer
    svm_explainer = shap.Explainer(
        best_svm_shap.predict_proba,
        X_train_shap
    )

    svm_shap_values = svm_explainer(X_test_shap)

    # Keep SHAP values for class 1
    if len(svm_shap_values.values.shape) == 3:
        svm_values_class1 = svm_shap_values.values[:, :, 1]
    else:
        svm_values_class1 = svm_shap_values.values

    SHAP_results["SVM"].append(svm_values_class1)
    SHAP_X_test["SVM"].append(X_test_shap.copy())


    # ==========================================
    # Random Forest
    # ==========================================

    rf_grid_shap = GridSearchCV(
        estimator=rf_pipeline,
        param_grid=param_grid_rf,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1
    )

    rf_grid_shap.fit(X_train_shap, y_train_shap)

    best_rf_shap = rf_grid_shap.best_estimator_

    print("Random Forest fitted.")

    # SHAP explainer
    rf_explainer = shap.Explainer(
        best_rf_shap.predict_proba,
        X_train_shap
    )

    rf_shap_values = rf_explainer(X_test_shap)

    # Keep SHAP values for class 1
    if len(rf_shap_values.values.shape) == 3:
        rf_values_class1 = rf_shap_values.values[:, :, 1]
    else:
        rf_values_class1 = rf_shap_values.values

    SHAP_results["RF"].append(rf_values_class1)
    SHAP_X_test["RF"].append(X_test_shap.copy())


    # ==========================================
    # Logistic Regression
    # ==========================================

    logreg_grid_shap = GridSearchCV(
        estimator=logreg_pipeline,
        param_grid=param_grid_logreg,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1
    )

    logreg_grid_shap.fit(X_train_shap, y_train_shap)

    best_logreg_shap = logreg_grid_shap.best_estimator_

    print("Logistic Regression fitted.")

    # SHAP explainer
    logreg_explainer = shap.Explainer(
        best_logreg_shap.predict_proba,
        X_train_shap
    )

    logreg_shap_values = logreg_explainer(X_test_shap)

    # Keep SHAP values for class 1
    if len(logreg_shap_values.values.shape) == 3:
        logreg_values_class1 = logreg_shap_values.values[:, :, 1]
    else:
        logreg_values_class1 = logreg_shap_values.values

    SHAP_results["Logistic Regression"].append(logreg_values_class1)
    SHAP_X_test["Logistic Regression"].append(X_test_shap.copy())


print("\n==========================================")
print("SHAP analysis completed for all 20 iterations.")
print("==========================================")

In [ ]:
import numpy as np
import pandas as pd

feature_names = X.columns.tolist()

SHAP_summary = {}

for model_name in ["SVM", "RF", "Logistic Regression"]:
    
    iteration_importances = []
    
    for i in range(20):
        
        # SHAP values for this iteration
        shap_values_i = SHAP_results[model_name][i]
        
        # Mean absolute SHAP value for each feature
        mean_abs_shap = np.mean(np.abs(shap_values_i), axis=0)
        
        iteration_importances.append(mean_abs_shap)
    
    # Convert to DataFrame
    iteration_importances = pd.DataFrame(
        iteration_importances,
        columns=feature_names
    )
    
    # Mean and SD across the 20 iterations
    summary = pd.DataFrame({
        "Feature": feature_names,
        "Mean_abs_SHAP": iteration_importances.mean(axis=0).values,
        "SD_abs_SHAP": iteration_importances.std(axis=0, ddof=1).values
    })
    
    # Sort by mean absolute SHAP
    summary = summary.sort_values(
        "Mean_abs_SHAP",
        ascending=False
    ).reset_index(drop=True)
    
    SHAP_summary[model_name] = summary


print("SVM")
print(SHAP_summary["SVM"])

print("\nRandom Forest")
print(SHAP_summary["RF"])

print("\nLogistic Regression")
print(SHAP_summary["Logistic Regression"])

In [ ]:
import matplotlib.pyplot as plt

models = ["SVM", "RF", "Logistic Regression"]

for model_name in models:

    plot_df = SHAP_summary[model_name].sort_values(
        "Mean_abs_SHAP",
        ascending=True
    )

    plt.figure(figsize=(8, 6))

    plt.barh(
        plot_df["Feature"],
        plot_df["Mean_abs_SHAP"],
        xerr=plot_df["SD_abs_SHAP"],
        capsize=4
    )

    plt.xlabel("Mean |SHAP value|")
    plt.ylabel("Feature")
    plt.title(f"{model_name} - SHAP Feature Importance")

    plt.tight_layout()
    plt.show()

In [ ]:
SVM_SHAP_all = np.vstack(SHAP_results["SVM"])
RF_SHAP_all = np.vstack(SHAP_results["RF"])
LogReg_SHAP_all = np.vstack(SHAP_results["Logistic Regression"])

SVM_X_all = pd.concat(SHAP_X_test["SVM"], axis=0)
RF_X_all = pd.concat(SHAP_X_test["RF"], axis=0)
LogReg_X_all = pd.concat(SHAP_X_test["Logistic Regression"], axis=0)

print("SVM:", SVM_SHAP_all.shape, SVM_X_all.shape)
print("RF:", RF_SHAP_all.shape, RF_X_all.shape)
print("Logistic Regression:", LogReg_SHAP_all.shape, LogReg_X_all.shape)

In [ ]:
print("SVM :")


shap.summary_plot(
    SVM_SHAP_all,
    SVM_X_all,
    feature_names=X.columns,
    show=True
)

In [ ]:
print("RF :")

shap.summary_plot(
    RF_SHAP_all,
    RF_X_all,
    feature_names=X.columns,
    show=True
)

In [ ]:
print ("LOGREG :")


shap.summary_plot(
    LogReg_SHAP_all,
    LogReg_X_all,
    feature_names=X.columns,
    show=True
)

In [ ]:
shap_final_table = pd.DataFrame({
    "Feature": feature_names
})

for model_name, column_prefix in [
    ("SVM", "SVM"),
    ("RF", "Random Forest"),
    ("Logistic Regression", "Logistic Regression")
]:
    
    temp = SHAP_summary[model_name].set_index("Feature")
    
    shap_final_table[f"{column_prefix} Mean ± SD"] = [
        f"{temp.loc[feature, 'Mean_abs_SHAP']:.4f} ± "
        f"{temp.loc[feature, 'SD_abs_SHAP']:.4f}"
        for feature in feature_names
    ]

print(shap_final_table.to_string(index=False))

In [ ]:
from scipy.stats import spearmanr

directionality_svm = []

for feature in feature_names:
    
    feature_values = SVM_X_all[feature].values
    
    feature_index = feature_names.index(feature)
    
    shap_values = SVM_SHAP_all[:, feature_index]
    
    rho, p_value = spearmanr(
        feature_values,
        shap_values
    )
    
    directionality_svm.append({
        "Feature": feature,
        "Spearman_rho": rho,
        "p_value": p_value
    })

directionality_svm = pd.DataFrame(directionality_svm)

print(directionality_svm)

In [ ]:
directionality_rf = []

for feature in feature_names:
    
    feature_values = RF_X_all[feature].values
    
    feature_index = feature_names.index(feature)
    
    shap_values = RF_SHAP_all[:, feature_index]
    
    rho, p_value = spearmanr(
        feature_values,
        shap_values
    )
    
    directionality_rf.append({
        "Feature": feature,
        "Spearman_rho": rho,
        "p_value": p_value
    })

directionality_rf = pd.DataFrame(directionality_rf)

print(directionality_rf)

In [ ]:
directionality_logreg = []

for feature in feature_names:
    
    feature_values = LogReg_X_all[feature].values
    
    feature_index = feature_names.index(feature)
    
    shap_values = LogReg_SHAP_all[:, feature_index]
    
    rho, p_value = spearmanr(
        feature_values,
        shap_values
    )
    
    directionality_logreg.append({
        "Feature": feature,
        "Spearman_rho": rho,
        "p_value": p_value
    })

directionality_logreg = pd.DataFrame(directionality_logreg)

print(directionality_logreg)

In [ ]:
directionality_table = pd.DataFrame({
    "Feature": feature_names,

    "SVM Spearman rho": [
        directionality_svm.set_index("Feature").loc[feature, "Spearman_rho"]
        for feature in feature_names
    ],

    "RF Spearman rho": [
        directionality_rf.set_index("Feature").loc[feature, "Spearman_rho"]
        for feature in feature_names
    ],

    "Logistic Regression Spearman rho": [
        directionality_logreg.set_index("Feature").loc[feature, "Spearman_rho"]
        for feature in feature_names
    ]
})

print(directionality_table.to_string(index=False))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


# =====================================
# Συνολικά Confusion Matrices - 20 iterations
# =====================================

total_cm_svm = np.sum(cms_svm, axis=0)
total_cm_rf = np.sum(cms_rf, axis=0)
total_cm_logreg = np.sum(cms_logreg, axis=0)


labels = ["None-Endpoint", "Endpoint"]


# =====================================
# SVM
# =====================================

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=total_cm_svm,
    display_labels=labels
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title("Confusion Matrix - SVM (20 iterations)")

plt.tight_layout()
plt.show()


# =====================================
# Random Forest
# =====================================

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=total_cm_rf,
    display_labels=labels
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title("Confusion Matrix - Random Forest (20 iterations)")

plt.tight_layout()
plt.show()


# =====================================
# Logistic Regression
# =====================================

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=total_cm_logreg,
    display_labels=labels
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title("Confusion Matrix - Logistic Regression (20 iterations)")

plt.tight_layout()

plt.savefig(
    "Figure_4_Confusion_Matrices.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Reduced model: remove the 2 crescent variables
X_reduced = X.iloc[:, :-2]

print("Full model features:", X.shape[1])
print("Reduced model features:", X_reduced.shape[1])

In [ ]:
Results_svm_reduced = []
Results_RF_reduced = []
Results_logreg_reduced = []

cms_svm_reduced = []
cms_rf_reduced = []
cms_logreg_reduced = []


for i in range(20):

    X_train, X_test, y_train, y_test = train_test_split(
        X_reduced,
        y,
        test_size=0.20,
        stratify=y,
        random_state=i
    )

    for model in models:

        if model == "SVM":

            svm_grid = GridSearchCV(
                estimator=svm_pipeline,
                param_grid=param_grid_svm,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1
            )

            svm_grid.fit(X_train, y_train)

            best_svm = svm_grid.best_estimator_

            y_pred_svm = best_svm.predict(X_test)

            cms_svm_reduced.append(
                confusion_matrix(y_test, y_pred_svm)
            )

            y_prob_svm = best_svm.predict_proba(X_test)[:, 1]

            accuracy = accuracy_score(y_test, y_pred_svm)
            precision = precision_score(
                y_test, y_pred_svm, zero_division=0
            )
            recall = recall_score(
                y_test, y_pred_svm, zero_division=0
            )
            f1 = f1_score(
                y_test, y_pred_svm, zero_division=0
            )
            roc_auc = roc_auc_score(
                y_test, y_prob_svm
            )
            pr_auc = average_precision_score(
                y_test, y_prob_svm
            )

            cm = confusion_matrix(
                y_test, y_pred_svm
            )

            TN, FP, FN, TP = cm.ravel()

            specificity = TN / (TN + FP)

            Results_svm_reduced.append({
                "iteration": i + 1,
                "Accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc-auc": roc_auc,
                "pr-auc": pr_auc,
                "Specificity": specificity
            })


        elif model == "RF":

            rf_grid = GridSearchCV(
                estimator=rf_pipeline,
                param_grid=param_grid_rf,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1
            )

            rf_grid.fit(X_train, y_train)

            best_rf = rf_grid.best_estimator_

            y_pred_rf = best_rf.predict(X_test)

            cms_rf_reduced.append(
                confusion_matrix(y_test, y_pred_rf)
            )

            y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

            accuracy = accuracy_score(y_test, y_pred_rf)
            precision = precision_score(
                y_test, y_pred_rf, zero_division=0
            )
            recall = recall_score(
                y_test, y_pred_rf, zero_division=0
            )
            f1 = f1_score(
                y_test, y_pred_rf, zero_division=0
            )
            roc_auc = roc_auc_score(
                y_test, y_prob_rf
            )
            pr_auc = average_precision_score(
                y_test, y_prob_rf
            )

            cm = confusion_matrix(
                y_test, y_pred_rf
            )

            TN, FP, FN, TP = cm.ravel()

            specificity = TN / (TN + FP)

            Results_RF_reduced.append({
                "iteration": i + 1,
                "Accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc-auc": roc_auc,
                "pr-auc": pr_auc,
                "Specificity": specificity
            })


        elif model == "Logistic Regression":

            logreg_grid = GridSearchCV(
                estimator=logreg_pipeline,
                param_grid=param_grid_logreg,
                scoring="roc_auc",
                cv=5,
                n_jobs=-1
            )

            logreg_grid.fit(X_train, y_train)

            best_logreg = logreg_grid.best_estimator_

            y_pred_logreg = best_logreg.predict(X_test)

            cms_logreg_reduced.append(
                confusion_matrix(y_test, y_pred_logreg)
            )

            y_prob_logreg = best_logreg.predict_proba(X_test)[:, 1]

            accuracy = accuracy_score(y_test, y_pred_logreg)
            precision = precision_score(
                y_test, y_pred_logreg, zero_division=0
            )
            recall = recall_score(
                y_test, y_pred_logreg, zero_division=0
            )
            f1 = f1_score(
                y_test, y_pred_logreg, zero_division=0
            )
            roc_auc = roc_auc_score(
                y_test, y_prob_logreg
            )
            pr_auc = average_precision_score(
                y_test, y_prob_logreg
            )

            cm = confusion_matrix(
                y_test, y_pred_logreg
            )

            TN, FP, FN, TP = cm.ravel()

            specificity = TN / (TN + FP)

            Results_logreg_reduced.append({
                "iteration": i + 1,
                "Accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc-auc": roc_auc,
                "pr-auc": pr_auc,
                "Specificity": specificity
            })

            
            
            
df_svm_reduced = pd.DataFrame(Results_svm_reduced)
df_RF_reduced = pd.DataFrame(Results_RF_reduced)
df_logreg_reduced = pd.DataFrame(Results_logreg_reduced)

total_cm_svm_reduced = np.sum(cms_svm_reduced, axis=0)
total_cm_rf_reduced = np.sum(cms_rf_reduced, axis=0)
total_cm_logreg_reduced = np.sum(cms_logreg_reduced, axis=0)

print("SVM Reduced:")
print(df_svm_reduced)

print("\nRandom Forest Reduced:")
print(df_RF_reduced)

print("\nLogistic Regression Reduced:")
print(df_logreg_reduced)

print("\nSVM Reduced Confusion Matrix:")
print(total_cm_svm_reduced)

print("\nRandom Forest Reduced Confusion Matrix:")
print(total_cm_rf_reduced)

print("\nLogistic Regression Reduced Confusion Matrix:")
print(total_cm_logreg_reduced)

In [ ]:
# ============================================
# REDUCED MODEL — MEAN ± SD
# ============================================

metrics = [
    "Accuracy",
    "precision",
    "recall",
    "Specificity",
    "f1",
    "roc-auc",
    "pr-auc"
]

summary_reduced = []

for model_name, df_model in [
    ("SVM", df_svm_reduced),
    ("Random Forest", df_RF_reduced),
    ("Logistic Regression", df_logreg_reduced)
]:

    row = {"Model": model_name}

    for metric in metrics:

        mean = df_model[metric].mean()
        sd = df_model[metric].std()

        row[metric] = f"{mean:.3f} ± {sd:.3f}"

    summary_reduced.append(row)


summary_reduced_df = pd.DataFrame(summary_reduced)

print(summary_reduced_df)

In [ ]:
# ============================================
# REDUCED MODEL — MEAN AND SD
# ============================================

mean_results_svm_reduced = df_svm_reduced.mean(numeric_only=True)
std_results_svm_reduced = df_svm_reduced.std(numeric_only=True)

mean_results_RF_reduced = df_RF_reduced.mean(numeric_only=True)
std_results_RF_reduced = df_RF_reduced.std(numeric_only=True)

mean_results_logreg_reduced = df_logreg_reduced.mean(numeric_only=True)
std_results_logreg_reduced = df_logreg_reduced.std(numeric_only=True)


# ============================================
# REDUCED COMPARISON TABLE
# ============================================

comparison_table_reduced = pd.DataFrame({

    "Model": [
        "SVM",
        "Random Forest",
        "Logistic Regression"
    ],

    "Accuracy": [
        f"{mean_results_svm_reduced['Accuracy']:.3f} ± {std_results_svm_reduced['Accuracy']:.3f}",
        f"{mean_results_RF_reduced['Accuracy']:.3f} ± {std_results_RF_reduced['Accuracy']:.3f}",
        f"{mean_results_logreg_reduced['Accuracy']:.3f} ± {std_results_logreg_reduced['Accuracy']:.3f}"
    ],

    "Precision": [
        f"{mean_results_svm_reduced['precision']:.3f} ± {std_results_svm_reduced['precision']:.3f}",
        f"{mean_results_RF_reduced['precision']:.3f} ± {std_results_RF_reduced['precision']:.3f}",
        f"{mean_results_logreg_reduced['precision']:.3f} ± {std_results_logreg_reduced['precision']:.3f}"
    ],

    "Recall": [
        f"{mean_results_svm_reduced['recall']:.3f} ± {std_results_svm_reduced['recall']:.3f}",
        f"{mean_results_RF_reduced['recall']:.3f} ± {std_results_RF_reduced['recall']:.3f}",
        f"{mean_results_logreg_reduced['recall']:.3f} ± {std_results_logreg_reduced['recall']:.3f}"
    ],

    "Specificity": [
        f"{mean_results_svm_reduced['Specificity']:.3f} ± {std_results_svm_reduced['Specificity']:.3f}",
        f"{mean_results_RF_reduced['Specificity']:.3f} ± {std_results_RF_reduced['Specificity']:.3f}",
        f"{mean_results_logreg_reduced['Specificity']:.3f} ± {std_results_logreg_reduced['Specificity']:.3f}"
    ],

    "F1": [
        f"{mean_results_svm_reduced['f1']:.3f} ± {std_results_svm_reduced['f1']:.3f}",
        f"{mean_results_RF_reduced['f1']:.3f} ± {std_results_RF_reduced['f1']:.3f}",
        f"{mean_results_logreg_reduced['f1']:.3f} ± {std_results_logreg_reduced['f1']:.3f}"
    ],

    "ROC-AUC": [
        f"{mean_results_svm_reduced['roc-auc']:.3f} ± {std_results_svm_reduced['roc-auc']:.3f}",
        f"{mean_results_RF_reduced['roc-auc']:.3f} ± {std_results_RF_reduced['roc-auc']:.3f}",
        f"{mean_results_logreg_reduced['roc-auc']:.3f} ± {std_results_logreg_reduced['roc-auc']:.3f}"
    ],

    "PR-AUC": [
        f"{mean_results_svm_reduced['pr-auc']:.3f} ± {std_results_svm_reduced['pr-auc']:.3f}",
        f"{mean_results_RF_reduced['pr-auc']:.3f} ± {std_results_RF_reduced['pr-auc']:.3f}",
        f"{mean_results_logreg_reduced['pr-auc']:.3f} ± {std_results_logreg_reduced['pr-auc']:.3f}"
    ]
})


comparison_table_reduced

In [ ]:
# ============================================
# FULL vs REDUCED — ITERATION-BY-ITERATION
# ============================================

comparison_differences = pd.DataFrame({

    "SVM_ROC_AUC_diff": (
        df_svm["roc-auc"].values -
        df_svm_reduced["roc-auc"].values
    ),

    "SVM_PR_AUC_diff": (
        df_svm["pr-auc"].values -
        df_svm_reduced["pr-auc"].values
    ),

    "RF_ROC_AUC_diff": (
        df_RF["roc-auc"].values -
        df_RF_reduced["roc-auc"].values
    ),

    "RF_PR_AUC_diff": (
        df_RF["pr-auc"].values -
        df_RF_reduced["pr-auc"].values
    ),

    "LogReg_ROC_AUC_diff": (
        df_logreg["roc-auc"].values -
        df_logreg_reduced["roc-auc"].values
    ),

    "LogReg_PR_AUC_diff": (
        df_logreg["pr-auc"].values -
        df_logreg_reduced["pr-auc"].values
    )
})

print(comparison_differences)

In [ ]:
# Mean ± SD of Full - Reduced differences

print("\nMean ± SD differences (Full - Reduced):\n")

for column in comparison_differences.columns:

    mean_diff = comparison_differences[column].mean()
    sd_diff = comparison_differences[column].std()

    print(
        f"{column}: "
        f"{mean_diff:.3f} ± {sd_diff:.3f}"
    )

In [ ]:
print("\nNumber of iterations where Full > Reduced:\n")

for column in comparison_differences.columns:

    count = (comparison_differences[column] > 0).sum()

    print(
        f"{column}: {count}/20"
    )

In [ ]:
from scipy.stats import t
import numpy as np


# ============================================
# NADEAU-BENGIO CORRECTED PAIRED T-TEST
# ============================================

def corrected_ttest(differences, n_train, n_test):

    differences = np.array(differences)

    n = len(differences)

    mean_diff = np.mean(differences)

    variance = np.var(differences, ddof=1)

    corrected_variance = variance * (
        (1 / n) + (n_test / n_train)
    )

    corrected_se = np.sqrt(corrected_variance)

    t_stat = mean_diff / corrected_se

    df = n - 1

    p_value = 2 * (1 - t.cdf(abs(t_stat), df))

    return mean_diff, corrected_se, t_stat, p_value

In [ ]:
n_total = len(y)

n_test = round(n_total * 0.20)
n_train = n_total - n_test

print("n_total:", n_total)
print("n_train:", n_train)
print("n_test:", n_test)

In [ ]:
# ============================================
# STATISTICAL COMPARISON
# FULL vs REDUCED
# ============================================

statistical_results = []

for column in comparison_differences.columns:

    differences = comparison_differences[column].values

    mean_diff, corrected_se, t_stat, p_value = corrected_ttest(
        differences,
        n_train,
        n_test
    )

    statistical_results.append({
        "Comparison": column,
        "Mean Difference": mean_diff,
        "Corrected SE": corrected_se,
        "t-statistic": t_stat,
        "p-value": p_value
    })


statistical_results_df = pd.DataFrame(
    statistical_results
)

print(statistical_results_df)

In [ ]:
print(df.shape)

print("\nOutcome:")
print(df["The combined endpoint"].value_counts())

In [ ]:
features = [
    "Baseline eGFR",
    "Serum creatine",
    "Triglyceride",
    "Proteinuria",
    "MAP",
    "Hematuria red blood cells/HPF",
    "Serum C3",
    "Age",
    "Crescent proportion",
    "Global crescent proportion"
]

print(df[features].describe())

In [ ]:
descriptive_results = []

for feature in features:
    
    class_0 = df.loc[
        df["The combined endpoint"] == 0, feature
    ].dropna()
    
    class_1 = df.loc[
        df["The combined endpoint"] == 1, feature
    ].dropna()

    descriptive_results.append({
        "Feature": feature,
        
        "None-Endpoint Median": class_0.median(),
        "None-Endpoint IQR": class_0.quantile(0.75) - class_0.quantile(0.25),
        
        "Endpoint Median": class_1.median(),
        "Endpoint IQR": class_1.quantile(0.75) - class_1.quantile(0.25)
    })

descriptive_df = pd.DataFrame(descriptive_results)

print(descriptive_df)

In [ ]:
table1_results = []

for feature in features:

    overall = df[feature].dropna()

    class_0 = df.loc[
        df["The combined endpoint"] == 0, feature
    ].dropna()

    class_1 = df.loc[
        df["The combined endpoint"] == 1, feature
    ].dropna()

    overall_median = overall.median()
    overall_iqr = overall.quantile(0.75) - overall.quantile(0.25)

    median_0 = class_0.median()
    iqr_0 = class_0.quantile(0.75) - class_0.quantile(0.25)

    median_1 = class_1.median()
    iqr_1 = class_1.quantile(0.75) - class_1.quantile(0.25)

    table1_results.append({
        "Variable": feature,
        "Overall (N=374)": f"{overall_median:.2f} ({overall_iqr:.2f})",
        "None-Endpoint (N=308)": f"{median_0:.2f} ({iqr_0:.2f})",
        "Endpoint (N=66)": f"{median_1:.2f} ({iqr_1:.2f})"
    })

table1_df = pd.DataFrame(table1_results)

print(table1_df.to_string(index=False))

In [ ]:
from scipy.stats import mannwhitneyu

table1_results = []

for feature in features:

    overall = df[feature].dropna()

    class_0 = df.loc[
        df["The combined endpoint"] == 0, feature
    ].dropna()

    class_1 = df.loc[
        df["The combined endpoint"] == 1, feature
    ].dropna()

    # Overall
    overall_median = overall.median()
    overall_iqr = overall.quantile(0.75) - overall.quantile(0.25)

    # None-Endpoint
    median_0 = class_0.median()
    iqr_0 = class_0.quantile(0.75) - class_0.quantile(0.25)

    # Endpoint
    median_1 = class_1.median()
    iqr_1 = class_1.quantile(0.75) - class_1.quantile(0.25)

    # Mann-Whitney U test
    _, p_value = mannwhitneyu(
        class_0,
        class_1,
        alternative="two-sided"
    )

    table1_results.append({
        "Variable": feature,
        "Overall (N=374)": f"{overall_median:.2f} ({overall_iqr:.2f})",
        "None-Endpoint (N=308)": f"{median_0:.2f} ({iqr_0:.2f})",
        "Endpoint (N=66)": f"{median_1:.2f} ({iqr_1:.2f})",
        "P-value": p_value
    })

table1_df = pd.DataFrame(table1_results)

print(table1_df.to_string(index=False))

In [ ]:
shap_summary = {}

for model_name, shap_values, X_all in [
    ("SVM", SVM_SHAP_all, SVM_X_all),
    ("Random Forest", RF_SHAP_all, RF_X_all),
    ("Logistic Regression", LogReg_SHAP_all, LogReg_X_all)
]:
    
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    std_abs_shap = np.std(np.abs(shap_values), axis=0)

    shap_summary[model_name] = pd.DataFrame({
        "Feature": X_all.columns,
        "Mean_abs_SHAP": mean_abs_shap,
        "SD_abs_SHAP": std_abs_shap
    }).sort_values(
        "Mean_abs_SHAP",
        ascending=False
    )

    print(f"\n{model_name}")
    print(shap_summary[model_name].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 8)
)

models = [
    ("(A) SVM", shap_summary["SVM"]),
    ("(B) Random Forest", shap_summary["Random Forest"]),
    ("(C) Logistic Regression", shap_summary["Logistic Regression"])
]

for ax, (model_name, data) in zip(axes, models):

    data_plot = data.sort_values(
        "Mean_abs_SHAP",
        ascending=True
    )

    ax.barh(
        data_plot["Feature"],
        data_plot["Mean_abs_SHAP"],
        xerr=data_plot["SD_abs_SHAP"],
        capsize=3
    )

    ax.set_title(model_name, fontsize=12)

    ax.set_xlabel("Mean |SHAP value|", fontsize=10)

    ax.set_ylabel("")

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.5
    )

    ax.tick_params(
        axis="y",
        labelsize=9
    )

fig.suptitle(
    "SHAP Feature Importance Across Machine-Learning Models",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = {
    "SVM": (Results_svm, Results_svm_reduced),
    "Random Forest": (Results_RF, Results_RF_reduced),
    "Logistic Regression": (Results_logreg, Results_logreg_reduced)
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

metrics = [
    ("roc-auc", "(A) ROC-AUC"),
    ("pr-auc", "(B) PR-AUC")
]

model_names = list(models.keys())

for ax, (metric, metric_label) in zip(axes, metrics):

    for model_idx, model_name in enumerate(model_names):

        full_results, reduced_results = models[model_name]

        full_values = [
            result[metric]
            for result in full_results
        ]

        reduced_values = [
            result[metric]
            for result in reduced_results
        ]

        x_full = np.full(
            len(full_values),
            model_idx - 0.10
        )

        x_reduced = np.full(
            len(reduced_values),
            model_idx + 0.10
        )

        # Paired lines
        for i in range(len(full_values)):
            ax.plot(
                [x_full[i], x_reduced[i]],
                [full_values[i], reduced_values[i]],
                linewidth=0.8,
                alpha=0.25
            )

        # Full
        ax.scatter(
            x_full,
            full_values,
            marker="o",
            s=35,
            label="Full" if model_idx == 0 else None
        )

        # Reduced
        ax.scatter(
            x_reduced,
            reduced_values,
            marker="s",
            s=35,
            label="Reduced" if model_idx == 0 else None
        )

    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names)

    ax.set_xlabel("Model")
    ax.set_ylabel(metric_label.replace("(A) ", "").replace("(B) ", ""))

    ax.set_title(metric_label)

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.5
    )

# One common legend
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    frameon=True,
    bbox_to_anchor=(0.5, 0.98)
)

fig.suptitle(
    "Full vs Reduced Models Across 20 Repeated Evaluations",
    fontsize=15,
    y=1.04
)

plt.tight_layout()
plt.show()